# Geolocation Processing

This notebook processes a dataset to extract location information and convert it to geographic coordinates (latitude/longitude) using Ollama for location extraction and OpenStreetMap (Nominatim) for geocoding.

## 1. Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import requests
import os
import glob
import json
import time
from datetime import datetime
from pathlib import Path
import re
from typing import Optional, Dict, Tuple

from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [2]:
# pip install geopy folium requests

## 2. Load Latest Dataset from Static Folder

In [2]:

static_folder = Path("../../static") 

csv_files = list(static_folder.glob("*.csv"))

if not csv_files:
    print(f"No CSV files found in {static_folder.absolute()}")
else:
    latest_file = max(csv_files, key=lambda p: p.stat().st_mtime)
    print(f"Latest file found: {latest_file.name}")
    print(f"Modified: {datetime.fromtimestamp(latest_file.stat().st_mtime)}")
    
    df = pd.read_csv(latest_file)
    print(f"\nDataset shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print("\nFirst few rows:")
    print(df.head())

Latest file found: all_merged.csv
Modified: 2026-01-21 14:05:43.129817

Dataset shape: (2172, 15)
Columns: ['Date', 'ExtractedAction', 'ExtractedAge', 'ExtractedDate', 'ExtractedGender', 'ExtractedTime', 'KeywordExtracted', 'KeywordMatch', 'Location', 'RightWingRelated', 'SourceFile', 'Text', 'Title', 'Topic', 'URL']

First few rows:
         Date                   ExtractedAction ExtractedAge ExtractedDate  \
0  18.12.2025  ['schlagen', 'körperverletzung']           []    18.12.2025   
1  18.07.2019                                []       ['34']    18.07.2019   
2  26.07.2019                                []           []    26.07.2019   
3  19.07.2021                   ['beleidigung']       ['28']    19.07.2021   
4  11.10.2019                                []           []    11.10.2019   

  ExtractedGender ExtractedTime      KeywordExtracted          KeywordMatch  \
0        ['mann']            []  ['fremdenfeindlich']  ['fremdenfeindlich']   
1              []     ['22.00']   ['v

In [ ]:
output_dir = Path("./output")
output_dir.mkdir(exist_ok=True)
geocoded_file = output_dir / "geocoded_data.csv"

df_previous = None
if geocoded_file.exists():
    print(f"\n Loading previously geocoded data from: {geocoded_file}")
    df_previous = pd.read_csv(geocoded_file)
    print(f"  Found {len(df_previous)} previously geocoded rows")
    
    if 'URL' in df.columns and 'URL' in df_previous.columns:
        df = df.merge(df_previous[['URL', 'standardized_location', 'ollama_extraction', 'latitude', 'longitude']], 
                      on='URL', how='left', suffixes=('_new', ''))
        for col in ['standardized_location', 'ollama_extraction', 'latitude', 'longitude']:
            if col + '_new' in df.columns:
                df[col] = df[col].fillna(df[col + '_new'])
                df = df.drop(columns=[col + '_new'], errors='ignore')
    
    print(f"  After merging: {df['latitude'].notna().sum()} rows already have coordinates\n")
else:
    print(f"\n  No previous geocoding found at {geocoded_file}")
    print("  Starting fresh...\n")

## 3. Ollama Configuration

In [ ]:
def save_checkpoint(dataframe, message=""):
    output_dir = Path("./output")
    output_dir.mkdir(exist_ok=True)
    output_file = output_dir / "geocoded_data.csv"
    
    df_geocoded = dataframe[dataframe['latitude'].notna() & dataframe['longitude'].notna()].copy()
    
    if len(df_geocoded) > 0:
        output_columns = df_geocoded.columns.tolist()
        geocoding_cols = ['standardized_location', 'latitude', 'longitude', 'ollama_extraction']
        for col in geocoding_cols:
            if col in output_columns:
                output_columns.remove(col)
                output_columns.append(col)
        
        df_output = df_geocoded[output_columns]
        df_output.to_csv(output_file, index=False)
        print(f"  ✓ Checkpoint saved: {len(df_output)} rows with coordinates {message}")
    else:
        print(f"  (No coordinates yet to save) {message}")


In [3]:
OLLAMA_API_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "gemma3:latest" 
REQUEST_TIMEOUT = 30

def test_ollama_connection():
    try:
        response = requests.get("http://localhost:11434/api/tags", timeout=5)
        if response.status_code == 200:
            models = response.json().get('models', [])
            print(f"✓ Ollama is running with {len(models)} model(s):")
            for model in models:
                print(f"  - {model.get('name', 'Unknown')}")
            return True
        else:
            print("✗ Ollama returned an error status")
            return False
    except requests.exceptions.ConnectionError:
        print("✗ Cannot connect to Ollama. Make sure it's running on http://localhost:11434")
        return False
    except Exception as e:
        print(f"✗ Error testing Ollama: {e}")
        return False

print("Testing Ollama connection...")
ollama_available = test_ollama_connection()

Testing Ollama connection...
✓ Ollama is running with 6 model(s):
  - qwen3:1.7b
  - llava:latest
  - smollm2:latest
  - gemma3:latest
  - gemma2:2b
  - codellama:latest


## 4. Extract and Standardize Location Data with Ollama

In [ ]:
def standardize_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = re.sub(r'\s+', ' ', text) 
    return text

def extract_location_with_ollama(text: str, location_context: str = None, row_context: Dict = None) -> Optional[str]:
    if not ollama_available or not text or not isinstance(text, str):
        return None
    
    context_info = ""
    if location_context:
        context_info = f"Geographic context (city, district): {location_context}\n"
    
    prompt = f"""{context_info}Extract the MOST SPECIFIC location mentioned in this text.

PRIORITY - Look for:
1. Street names (e.g., "Ratsgasse", "Friedrich-Wolf-Ring", "Berliner Straße")
2. Landmarks, buildings, or specific places (e.g., "Bahnhof", "Bushaltestelle", "Schule")
3. Areas or neighborhoods
4. Addresses with numbers

Format: [Street/Place], [City], [District] (if available)

Examples of correct responses:
- "Ratsgasse, Velten, Oberhavel"
- "Mühlenweg 12, Strausberg, Märkisch-Oderland"
- "Bernauer Straße, Oranienburg, Oberhavel"
- "corner near Bahnhof, Neuruppin, Ostprignitz-Ruppin"

Return ONLY the location, nothing else. If no specific location found in text, use context: "{location_context}". If completely unknown, respond 'UNKNOWN'.

Text: "{text}"

Most specific location:"""
    
    try:
        response = requests.post(
            OLLAMA_API_URL,
            json={
                "model": OLLAMA_MODEL,
                "prompt": prompt,
                "stream": False
            },
            timeout=REQUEST_TIMEOUT
        )
        
        if response.status_code == 200:
            result = response.json()
            location = result.get("response", "").strip()
            location = standardize_text(location)
            
            if location and location.lower() != "unknown":
                if len(location) > 120:
                    location = location[:120].rsplit(',', 1)[0].strip()
                
                return location
    except Exception as e:
        print(f"Error querying Ollama: {e}")
    
    return None


if 'standardized_location' not in df.columns:
    df['standardized_location'] = None
if 'ollama_extraction' not in df.columns:
    df['ollama_extraction'] = None

already_extracted = df['standardized_location'].notna().sum()
print(f"\nExtracting precise, concise locations from {len(df)} rows...")
print(f"Already extracted: {already_extracted} locations")
print("Note: This may take a while depending on dataset size and Ollama response time.\n")


Extracting precise, concise locations from 2172 rows...
Note: This may take a while depending on dataset size and Ollama response time.



In [ ]:
sample_size = 600  
shuffle_data = True  

if shuffle_data:
    df_process = df.sample(frac=1, random_state=42).reset_index(drop=True)
    print(f"Processing {sample_size} rows out of {len(df)} total rows (shuffled)...\n")
else:
    df_process = df.copy()
    print(f"Processing {sample_size} rows out of {len(df)} total rows (sequential)...\n")

for idx, row in df_process.head(sample_size).iterrows():
    if pd.notna(row['standardized_location']):
        continue
    
    location_context = None
    if 'Location' in df_process.columns and pd.notna(row['Location']):
        location_context = str(row['Location'])
    
    location_text = " ".join([
        str(row[col]) for col in df_process.columns 
        if pd.notna(row[col]) and row[col] != ""
    ])
    
    if location_text:
        extracted_location = extract_location_with_ollama(location_text, location_context, dict(row))
        original_idx = df[df['URL'] == row.get('URL')].index[0] if 'URL' in df.columns and pd.notna(row.get('URL')) else None
        if original_idx is not None:
            df.at[original_idx, 'ollama_extraction'] = extracted_location
            df.at[original_idx, 'standardized_location'] = extracted_location
        
        if extracted_location:
            print(f"  Row {idx + 1}: {extracted_location} ({row.get('Location', 'Unknown')})")
        
        if (idx + 1) % 10 == 0:
            print(f"  Processed {idx + 1}/{sample_size} rows...")
            
        time.sleep(0.5)

print(f"\nLocation extraction complete!")
print(f"Successfully extracted: {df['standardized_location'].notna().sum()} locations")
print("Saving checkpoint...")
save_checkpoint(df, "(after location extraction)")
print("\nSample results:")
print(df[['standardized_location', 'ollama_extraction']].head(10))

Processing 600 rows out of 2172 total rows (SHUFFLED for diversity)...

  Row 1: Goltzstraße Ecke Hohenstaufenstraße, Tempelhof-Schöneberg (Tempelhof-Schöneberg)
  Row 2: Alte Knehder Straße, Templin, Uckermark (Templin, Uckermark)
  Row 3: Treptow, Berlin, Berlin (berlinweit)
  Row 4: Radweg zwischen Weggun und Krewitz, Uckermark (Nordwestuckermark/Boitzenburger Land, Uckermark)
  Row 5: Mühlenweg, Schönfließ, Oberhavel (Schönfließ, Oberhavel)
  Row 6: In der Straße des Friedens, Gransee, Oberhavel (Gransee, Oberhavel)
  Row 7: Logenstraße, Lübben, Dahme-Spreewald (Lübben, Dahme-Spreewald)
  Row 8: Brunsbütteler Damm, Spandau (Spandau)
  Row 9: Wiltbergstraße, Berlin, Pankow (Pankow)
  Row 10: Treidelweg, Eberswalde, Barnim (Eberswalde, Barnim)
  Processed 10/600 rows...
  Row 11: Wartenberger Straße, Lichtenberg (Lichtenberg)
  Row 12: Märkische Allee, Marzahn-Hellersdorf (Marzahn-Hellersdorf)
  Row 13: U-Bahnhof Theodor-Heuss-Platz, Charlottenburg-Wilmersdorf (Charlottenburg-Wilmers

## 5. Geocode Locations with OpenStreetMap

In [ ]:
geocoder = Nominatim(user_agent="geolocation_processor")

def geocode_location(location_string: str) -> Optional[Tuple[float, float]]:
   
    if not location_string or not isinstance(location_string, str):
        return None
    
    try:
        location = geocoder.geocode(location_string, timeout=10)
        if location:
            return (location.latitude, location.longitude)
    except GeocoderTimedOut:
        print(f"  Timeout geocoding: {location_string}")
    except GeocoderServiceError as e:
        print(f"  Service error geocoding {location_string}: {e}")
    except Exception as e:
        print(f"  Error geocoding {location_string}: {e}")
    
    return None

# Initialize coordinates if they don't exist
if 'latitude' not in df.columns:
    df['latitude'] = None
if 'longitude' not in df.columns:
    df['longitude'] = None

# Count already geocoded
already_geocoded = df[df['latitude'].notna() & df['longitude'].notna()].shape[0]
to_geocode = df[(df['standardized_location'].notna()) & (df['latitude'].isna())].shape[0]

print(f"Geocoding locations...")
print(f"  Already geocoded: {already_geocoded}")
print(f"  To geocode: {to_geocode}\n")

geocoded_count = 0
for idx, row in df.iterrows():
    if pd.notna(row['latitude']) and pd.notna(row['longitude']):
        continue
    
    if pd.notna(row['standardized_location']):
        coords = geocode_location(row['standardized_location'])
        if coords:
            df.at[idx, 'latitude'] = coords[0]
            df.at[idx, 'longitude'] = coords[1]
            geocoded_count += 1
        
        if (idx + 1) % 10 == 0:
            print(f"  Processed {idx + 1} rows ({geocoded_count} newly geocoded)...")
        
        # Save checkpoint every 50 rows geocoded
        if geocoded_count > 0 and geocoded_count % 50 == 0:
            save_checkpoint(df, f"(after {geocoded_count} geocoded)")
        
        time.sleep(1)

print(f"\nFirst pass complete: {geocoded_count} locations geocoded")
print("Saving checkpoint...")
save_checkpoint(df, "(after first geocoding pass)")

# Fallback for Location field
remaining = df[(df['latitude'].isna()) & (df['Location'].notna())].shape[0]
if remaining > 0:
    print(f"\nGeocoding {remaining} Location fields (fallback)...")
    fallback_count = 0
    
    for idx, row in df.iterrows():
        if pd.notna(row['latitude']) and pd.notna(row['longitude']):
            continue
        
        if pd.notna(row['Location']):
            coords = geocode_location(str(row['Location']))
            if coords:
                df.at[idx, 'latitude'] = coords[0]
                df.at[idx, 'longitude'] = coords[1]
                fallback_count += 1
            
            if (idx + 1) % 10 == 0:
                print(f"  Processed {idx + 1} fallback rows ({fallback_count} successful)...")
            
            # Save checkpoint every 30 rows
            if fallback_count > 0 and fallback_count % 30 == 0:
                save_checkpoint(df, f"(after {fallback_count} fallback geocoded)")
            
            time.sleep(1)
    
    print(f"Fallback geocoding: {fallback_count} additional locations")
    print("Saving checkpoint...")
    save_checkpoint(df, "(after fallback geocoding)")
else:
    fallback_count = 0

print(f"\nGeocoding complete!")
print(f"Total newly geocoded this run: {geocoded_count + fallback_count}")
print(f"Total successfully geocoded overall: {df[df['latitude'].notna()].shape[0]}")
print("\nSample results with coordinates:")
print(df[df['latitude'].notna()][['standardized_location', 'Location', 'latitude', 'longitude']].head(10))

Geocoding 600 standardized locations...
  Geocoded 30 rows... (3 successful)
  Geocoded 40 rows... (6 successful)
  Geocoded 50 rows... (8 successful)
  Geocoded 60 rows... (11 successful)
  Geocoded 70 rows... (13 successful)
  Geocoded 100 rows... (18 successful)
  Geocoded 110 rows... (20 successful)
  Geocoded 210 rows... (40 successful)
  Geocoded 240 rows... (43 successful)
  Geocoded 260 rows... (48 successful)
  Geocoded 290 rows... (52 successful)
  Geocoded 300 rows... (56 successful)
  Geocoded 410 rows... (78 successful)
  Geocoded 430 rows... (83 successful)
  Geocoded 480 rows... (97 successful)
  Geocoded 520 rows... (104 successful)
  Geocoded 530 rows... (105 successful)
  Geocoded 580 rows... (118 successful)
  Geocoded 600 rows... (122 successful)
  Geocoded 650 rows... (135 successful)
  Geocoded 680 rows... (137 successful)
  Geocoded 720 rows... (141 successful)
  Geocoded 760 rows... (149 successful)
  Geocoded 820 rows... (162 successful)
  Geocoded 840 rows... 

## 6. Store Results and Geocoordinates

In [ ]:
output_dir = Path("./output")
output_dir.mkdir(exist_ok=True)
output_file = output_dir / "geocoded_data.csv"

df_geocoded = df[df['latitude'].notna() & df['longitude'].notna()].copy()

output_columns = df_geocoded.columns.tolist()
geocoding_cols = ['standardized_location', 'latitude', 'longitude', 'ollama_extraction']
for col in geocoding_cols:
    if col in output_columns:
        output_columns.remove(col)
        output_columns.append(col)

df_output = df_geocoded[output_columns]
df_output.to_csv(output_file, index=False)
print(f"✓ Results saved to: {output_file}")
print(f"  Total rows with coordinates: {len(df_output)}")
print(f"  Rows skipped (no geocoding): {len(df) - len(df_output)}")

print("\n" + "="*60)
print("GEOCODING SUMMARY")
print("="*60)
print(f"Total locations extracted: {df['standardized_location'].notna().sum()}")
print(f"Successfully geocoded: {df_geocoded.shape[0]}")
print(f"Success rate: {(len(df_output) / df['standardized_location'].notna().sum() * 100):.1f}%" if df['standardized_location'].notna().sum() > 0 else "No locations found")

if len(df_output) > 0:
    print("\nCoordinate ranges:")
    print(f"  Latitude: {df_output['latitude'].min():.4f} to {df_output['latitude'].max():.4f}")
    print(f"  Longitude: {df_output['longitude'].min():.4f} to {df_output['longitude'].max():.4f}")

print("\nSample geocoded results:")
print(df_output[['standardized_location', 'latitude', 'longitude']].head(10))

✓ Results saved to: output/geocoded_data_20260121_162421.csv
  Total rows with coordinates: 883
  Rows skipped (no geocoding): 1289

GEOCODING SUMMARY
Total locations processed: 600
Successfully geocoded: 883
Success rate: 147.2%

Coordinate ranges:
  Latitude: 49.3842 to 53.4096
  Longitude: 6.9537 to 14.7154

Sample geocoded results:
  standardized_location   latitude  longitude
0                  None  51.751299  14.322308
1                  None  52.924386  12.809292
2                  None  53.090801  11.474862
3                  None  51.588327  14.012314
4                  None  52.558894  13.904248
5                  None  52.752938  13.245759
6                  None   53.11935  13.500556
7                  None  52.752938  13.245759
8                  None  51.751299  14.322308
9                  None  52.835081  13.799654
